In [1]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from construct import *
import scipy.signal

import ccsds
import tianwen1_tm

import struct
import collections

plt.rcParams['figure.figsize'] = (7, 3.5)

In [2]:
def get_packet(p):
    return p[0] if type(p) is tuple else p

def packets_asarray(packets):
    return np.array([np.frombuffer(get_packet(p)[ccsds.SpacePacketPrimaryHeader.sizeof():], 'uint8')
                     for p in packets])

def plot_apids(apids, sc, vc):
    for apid in sorted(apids.keys()):
        plt.figure(figsize = (16,16), facecolor = 'w')
        ps = packets_asarray(apids[apid])
        plt.imshow(ps, aspect = ps.shape[1]/ps.shape[0])
        plt.title(f'APID {apid} Spacecraft {sc} Virtual channel {vc}')
        
def get_timestamps(aos):
    return np.array([tianwen1_tm.parse_timestamp_datetime64(a.insert_zone.timestamp) for a in aos])

def get_packet_timestamps(packets):
    return np.array([tianwen1_tm.parse_timestamp_datetime64(p[1]) for p in packets])

In [3]:
frame_size = 220
frames = np.fromfile('tianwen1_safe_mode_frames.u8', dtype = 'uint8')
frames = frames[:frames.size//frame_size*frame_size].reshape((-1, frame_size))
frames.shape[0]

2

In [4]:
aos = [ccsds.AOSFrame.parse(f) for f in frames]
[a.primary_header for a in aos]

[Container(transfer_frame_version_number=1, spacecraft_id=245, virtual_channel_id=1, virtual_channel_frame_count=6286530, replay_flag=False, vc_frame_count_usage_flag=False, rsvd_spare=0, vc_framecount_cycle=0),
 Container(transfer_frame_version_number=1, spacecraft_id=245, virtual_channel_id=1, virtual_channel_frame_count=6286531, replay_flag=False, vc_frame_count_usage_flag=False, rsvd_spare=0, vc_framecount_cycle=0)]

In [5]:
vc1 = list(ccsds.extract_space_packets(aos, 245, 1, get_timestamps = True))
vc1_headers = [ccsds.SpacePacketPrimaryHeader.parse(p[0]) for p in vc1]
vc1_apids = collections.Counter([p.APID for p in vc1_headers])
vc1_by_apid = {apid : [p for h,p in zip(vc1_headers, vc1)
                       if h.APID == apid] for apid in vc1_apids}
vc1_headers

[Container(ccsds_version=0, packet_type=0, secondary_header_flag=False, APID=1280, sequence_flags=3, packet_sequence_count_or_name=8730, data_length=57),
 Container(ccsds_version=0, packet_type=0, secondary_header_flag=False, APID=1536, sequence_flags=3, packet_sequence_count_or_name=13024, data_length=21),
 Container(ccsds_version=0, packet_type=0, secondary_header_flag=False, APID=1025, sequence_flags=3, packet_sequence_count_or_name=5765, data_length=57),
 Container(ccsds_version=0, packet_type=0, secondary_header_flag=False, APID=768, sequence_flags=3, packet_sequence_count_or_name=14322, data_length=11),
 Container(ccsds_version=0, packet_type=0, secondary_header_flag=False, APID=449, sequence_flags=3, packet_sequence_count_or_name=9204, data_length=15),
 Container(ccsds_version=0, packet_type=0, secondary_header_flag=False, APID=2, sequence_flags=3, packet_sequence_count_or_name=8149, data_length=11),
 Container(ccsds_version=0, packet_type=0, secondary_header_flag=False, APID=3,

In [6]:
get_packet_timestamps(vc1)

array(['2026-08-02T15:47:39.993600', '2026-08-02T15:47:39.993600',
       '2026-08-02T15:47:39.993600', '2026-08-02T15:47:39.993600',
       '2026-08-02T15:49:47.994000', '2026-08-02T15:49:47.994000',
       '2026-08-02T15:49:47.994000', '2026-08-02T15:49:47.994000',
       '2026-08-02T15:49:47.994000', '2026-08-02T15:49:47.994000'],
      dtype='datetime64[us]')

In [7]:
for apid in sorted(vc1_by_apid):
    print(f'APID {apid}')
    for p in vc1_by_apid[apid]:
        print(p[0].hex())
    print()

APID 2
0002dfd5000b9413ad01f8000000eaa9f552

APID 3
0003dfd5000baafcf601f80000001548f552

APID 14
000ec2b2002728020079d603f703fc03f78100000d200000008b0000090b0000c279fd9f00000000000fffff0000

APID 202
00cadaaf0039807f7f7f7f7f7ffffff4f4807ff4f47f7ff47ff48b7f7ff48b7f7f80808085857ff480807f7ff3f4f37f7f7f7f7f7f7f7feced8d98e6eae6ea7f

APID 449
01c1e3f4000f0309e7f697be0000e200202003315208

APID 768
0300f7f2000b08007c0000008c00f800f781

APID 832
0340e260000b6000560000f8fc02cafaf781

APID 1025
0401d68500390000503f503f503f0000000003cf08000000000000007fff0456001402ff0a25002d261234029f9a0f23fe00fe000f39edad86436296fe61d0d8

APID 1280
0500e21a0039000000000000000000000000000000000000d708cee2002a002b005a01f40064000a006e00d200500186ff9200090032000a0001000b0015e6fd

APID 1536
0600f2e00015a9a569f03f00800fb00e0001e65100438067b5f0eb60

